In [1]:
import pandas as pd
import numpy as np
import ast
import re
import math

df = pd.read_csv("coursera_courses.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 1000
Columns: 12


In [2]:
print(df.columns.tolist())

['course_title', 'course_organization', 'course_certificate_type', 'course_time', 'course_rating', 'course_reviews_num', 'course_difficulty', 'course_url', 'course_students_enrolled', 'course_skills', 'course_summary', 'course_description']


In [3]:
df.head()

,course_title,course_organization,course_certificate_type,course_time,course_rating,course_reviews_num,course_difficulty,course_url,course_students_enrolled,course_skills,course_summary,course_description
0,(ISC)² Systems Security Certified Practitioner...,ISC2,Specialization,3 - 6 Months,4.7,492,Beginner,https://www.coursera.org/specializations/sscp-...,"6,958","['Risk Management', 'Access Control', 'Asset',...",[],Pursue better IT security job opportunities an...
1,.NET FullStack Developer,Board Infinity,Specialization,1 - 3 Months,4.3,51,Intermediate,https://www.coursera.org/specializations/dot-n...,"2,531","['Web API', 'Web Development', 'Cascading Styl...",['Master .NET full stack web dev: from .NET co...,Develop the proficiency required to design and...
2,21st Century Energy Transition: how do we make...,University of Alberta,Course,1 - 3 Months,4.8,62,Beginner,https://www.coursera.org/learn/21st-century-en...,"4,377",[],['Understand the complexity of systems supplyi...,"Affordable, abundant and reliable energy is fu..."
3,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,Course,1 - 3 Months,4.7,517,Intermediate,https://www.coursera.org/learn/crash-course-in...,"39,004","['Instrumental Variable', 'Propensity Score Ma...",[],We have all heard the phrase “correlation does...
4,A life with ADHD,University of Geneva,Course,1 - 3 Months,NaN,NaN,Beginner,https://www.coursera.org/learn/life-with-adhd,NaN,"['differential diagnosis and comorbidities', '...",[' Understand what ADHD is and the challenges ...,What is ADHD and what are the challenges that ...


In [4]:
print(df.isnull().sum())

course_title                 0
course_organization          0
course_certificate_type      0
course_time                  0
course_rating                6
course_reviews_num           6
course_difficulty            0
course_url                   0
course_students_enrolled    41
course_skills                0
course_summary               0
course_description           1
dtype: int64


In [5]:
courses = df.copy()

In [6]:
courses.columns = (
    courses.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(courses.columns.tolist())

['course_title', 'course_organization', 'course_certificate_type', 'course_time', 'course_rating', 'course_reviews_num', 'course_difficulty', 'course_url', 'course_students_enrolled', 'course_skills', 'course_summary', 'course_description']


In [7]:
before = len(courses)

courses = courses.drop_duplicates(
    subset=["course_title"]
).reset_index(drop=True)

after = len(courses)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 1000
Rows after: 993
Duplicates removed: 7


In [8]:
courses["course_title"] = (
    courses["course_title"]
    .astype(str)
    .str.strip()
)


In [9]:
def clean_skill_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    
    text = re.sub(r"[\[\]{}'\"]", "", text)
    text = re.sub(r"\s+", " ", text)
    
    return text

In [10]:
courses["course_skills_clean"] = courses["course_skills"].apply(
    clean_skill_text
)

courses[["course_title", "course_skills", "course_skills_clean"]].head()

,course_title,course_skills,course_skills_clean
0,(ISC)² Systems Security Certified Practitioner...,"['Risk Management', 'Access Control', 'Asset',...","risk management, access control, asset, incide..."
1,.NET FullStack Developer,"['Web API', 'Web Development', 'Cascading Styl...","web api, web development, cascading style shee..."
2,21st Century Energy Transition: how do we make...,[],
3,A Crash Course in Causality: Inferring Causal...,"['Instrumental Variable', 'Propensity Score Ma...","instrumental variable, propensity score matchi..."
4,A life with ADHD,"['differential diagnosis and comorbidities', '...","differential diagnosis and comorbidities, symp..."


In [15]:
courses["course_rating"] = pd.to_numeric(
    courses["course_rating"],
    errors="coerce"
)

courses["course_rating"] = courses["course_rating"].fillna(
    courses["course_rating"].median()
)

In [14]:
courses["course_reviews_num"] = pd.to_numeric(
    courses["course_reviews_num"],
    errors="coerce"
)

courses["course_reviews_num"] = courses["course_reviews_num"].fillna(0)

In [13]:
courses["course_difficulty"] = (
    courses["course_difficulty"]
    .astype(str)
    .str.strip()
    .str.lower()
)

courses["course_difficulty"].value_counts()

course_difficulty
beginner        678
intermediate    199
mixed            80
advanced         36
Name: count, dtype: int64

In [16]:
# DIFFICULTY WEIGHT

##CourseRank=Rating×log(1+Reviews)×DifficultyWeight

difficulty_weights = {
    "beginner": 1.0,
    "intermediate": 1.1,
    "advanced": 1.2
}

In [17]:
courses["difficulty_weight"] = (
    courses["course_difficulty"]
    .map(difficulty_weights)
    .fillna(1.0)
)

courses[
    [
        "course_title",
        "course_difficulty",
        "difficulty_weight"
    ]
].head()

,course_title,course_difficulty,difficulty_weight
0,(ISC)² Systems Security Certified Practitioner...,beginner,1.0
1,.NET FullStack Developer,intermediate,1.1
2,21st Century Energy Transition: how do we make...,beginner,1.0
3,A Crash Course in Causality: Inferring Causal...,intermediate,1.1
4,A life with ADHD,beginner,1.0


In [18]:
courses["course_rank"] = (
    courses["course_rating"].astype(float)
    *
    np.log1p(courses["course_reviews_num"].astype(float))
    *
    courses["difficulty_weight"]
)


In [19]:
courses["course_rank"] = (
    courses["course_rating"].astype(float)
    *
    np.log1p(courses["course_reviews_num"].astype(float))
    *
    courses["difficulty_weight"]
)


In [20]:
##INVERTED SKILL-TO-COURSE INDEX

#Skill → Courses containing that skill
### Python
   #
def extract_skills(skill_text):
    if not skill_text:
        return []
    
    skills = re.split(r",|;|\||\n", skill_text)
    
    cleaned = []
    
    for skill in skills:
        skill = skill.strip().lower()
        
        if skill:
            cleaned.append(skill)
    
    return list(set(cleaned))

In [21]:
courses["skill_list"] = courses["course_skills_clean"].apply(
    extract_skills
)

courses[
    ["course_title", "skill_list"]
].head()

,course_title,skill_list
0,(ISC)² Systems Security Certified Practitioner...,"[cloud computing security, wireless security, ..."
1,.NET FullStack Developer,"[html, web development, cascading style sheets..."
2,21st Century Energy Transition: how do we make...,[]
3,A Crash Course in Causality: Inferring Causal...,"[instrumental variable, causality, causal infe..."
4,A life with ADHD,"[intervention, neuroscience, differential diag..."


In [23]:
## REGEX BOUNDARY MATCH
#We don't want:"java"
#to accidentally match:"javascript"

def skill_matches_course(skill, course_text):
    if not skill:
        return False
    
    skill = skill.lower().strip()
    course_text = course_text.lower()
    
    pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
    
    return bool(re.search(pattern, course_text))
    


In [31]:
inverted_index = {}

for index, row in courses.iterrows():

    course_text = (
        str(row["course_title"]) + " " +
        str(row["course_skills_clean"])
    ).lower()

    # Extract individual words from course text
    words = set(re.findall(r"\b[a-zA-Z0-9+#.]+\b", course_text))

    for word in words:

        if word not in inverted_index:
            inverted_index[word] = []

        inverted_index[word].append(index)

In [32]:
print("Number of unique skills:", len(inverted_index))


Number of unique skills: 2583


In [26]:
list(inverted_index.keys())[:20]

['cloud computing security',
 'wireless security',
 'incident detection and response',
 'access control',
 'asset',
 'security software',
 'risk management',
 'html',
 'web development',
 'cascading style sheets (css)',
 'model–view–controller (mvc)',
 'react (web framework)',
 'asp.net',
 'c# programming',
 'web api',
 'api integration',
 'restful apis',
 'instrumental variable',
 'causality',
 'causal inference']

In [33]:
skill = "python"

print("Python found in index:", skill in inverted_index)

if skill in inverted_index:
    print("Number of candidate courses:", len(inverted_index[skill]))

Python found in index: True
Number of candidate courses: 76


In [34]:
skill = "python"

candidate_indices = inverted_index.get(
    skill.lower(),
    []
)

matched_rows = []

pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"

for index in candidate_indices:

    course_text = (
        str(courses.loc[index, "course_title"]) + " " +
        str(courses.loc[index, "course_skills_clean"])
    ).lower()

    if re.search(pattern, course_text):
        matched_rows.append(index)


courses.loc[
    matched_rows,
    [
        "course_title",
        "course_rating",
        "course_reviews_num",
        "course_difficulty",
        "course_rank"
    ]
].sort_values(
    "course_rank",
    ascending=False
).head(10)

,course_title,course_rating,course_reviews_num,course_difficulty,course_rank
991,用 Python 做商管程式設計（一）(Programming for Business C...,4.9,814.0,beginner,32.845622
933,The Power of Statistics,4.8,270.0,advanced,32.268204
90,Automate Cybersecurity Tasks with Python,4.7,915.0,beginner,32.054077
91,"BI Foundations with SQL, ETL and Data Warehousing",4.8,665.0,beginner,31.206190
818,Python Project for Data Engineering,4.6,473.0,intermediate,31.175709
69,Applied Software Engineering Fundamentals,4.5,792.0,beginner,30.041204
830,Regression Analysis: Simplify Complex Data Rel...,4.7,175.0,advanced,29.161530
821,Python for Cybersecurity,4.5,328.0,intermediate,28.690486
473,IBM AI Enterprise Workflow,4.3,258.0,advanced,28.673233
536,Introducción a la programación en Python I: Ap...,4.6,496.0,beginner,28.559514


In [29]:
def recommend_courses(missing_skills, top_n=10):
    
    matched_indices = set()
    
    for skill in missing_skills:
        
        skill = skill.lower().strip()
        
        for index, row in courses.iterrows():
            
            course_text = (
                str(row["course_title"]) + " " +
                str(row["course_skills_clean"])
            ).lower()
            
            if skill_matches_course(skill, course_text):
                matched_indices.add(index)
    
    if not matched_indices:
        return pd.DataFrame()
    
    result = courses.loc[
        list(matched_indices)
    ].copy()
    
    result = result.sort_values(
        "course_rank",
        ascending=False
    )
    
    return result.head(top_n)

In [35]:
def recommend_courses(missing_skills, top_n=10):

    matched_indices = set()

    for skill in missing_skills:

        skill = skill.lower().strip()

        # Get candidate courses from inverted index
        candidate_indices = inverted_index.get(skill, [])

        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"

        for index in candidate_indices:

            course_text = (
                str(courses.loc[index, "course_title"]) + " " +
                str(courses.loc[index, "course_skills_clean"])
            ).lower()

            # Regex boundary verification
            if re.search(pattern, course_text):
                matched_indices.add(index)

    if not matched_indices:
        return pd.DataFrame()

    result = courses.loc[list(matched_indices)].copy()

    # Rank courses using CourseRank
    result = result.sort_values(
        "course_rank",
        ascending=False
    )

    return result.head(top_n)

In [36]:
missing_skills = [
    "python",
    "machine learning",
    "sql"
]

recommendations = recommend_courses(
    missing_skills,
    top_n=10
)

recommendations

,course_title,course_organization,course_certificate_type,course_time,course_rating,course_reviews_num,course_difficulty,course_url,course_students_enrolled,course_skills,course_summary,course_description,course_skills_clean,difficulty_weight,course_rank,skill_list
272,Django for Everybody,University of Michigan,Specialization,3 - 6 Months,4.7,904.0,intermediate,https://www.coursera.org/specializations/django,"50,277","['Cascading Style Sheets (CSS)', 'HTML', 'Java...",[],Diversity is a fact. It is also paradoxical. W...,"cascading style sheets (css), html, javascript...",1.1,35.197024,"[javascript, html, cascading style sheets (css..."
991,用 Python 做商管程式設計（一）(Programming for Business C...,National Taiwan University,Course,1 - 3 Months,4.9,814.0,beginner,https://www.coursera.org/learn/pbc1,"38,595",[],[],本系列課程從零開始，教授一般認為最適合初學者的程式語言「Python」，目標是讓大家在完成本...,,1.0,32.845622,[]
933,The Power of Statistics,Google,Course,1 - 3 Months,4.8,270.0,advanced,https://www.coursera.org/learn/the-power-of-st...,"22,700","['Statistical Analysis', 'Python Programming',...","['Explore and summarize a dataset ', 'Use prob...",This is the fourth of seven courses in the Goo...,"statistical analysis, python programming, effe...",1.2,32.268204,"[python programming, statistical analysis, eff..."
90,Automate Cybersecurity Tasks with Python,Google,Course,1 - 4 Weeks,4.7,915.0,beginner,https://www.coursera.org/learn/automate-cybers...,"68,630","['Computer Programming', 'Python Programming',...","['Work with architectural and site elements, e...",Prove to potential employers that you’re up to...,"computer programming, python programming, codi...",1.0,32.054077,"[computer programming, pep 8 style guide, codi..."
91,"BI Foundations with SQL, ETL and Data Warehousing",IBM,Specialization,3 - 6 Months,4.8,665.0,beginner,https://www.coursera.org/specializations/bi-fo...,"56,137","['Cloud Databases', 'Python Programming', 'Jup...",['Explain how the Python programming language ...,This is the seventh course in the Google Cyber...,"cloud databases, python programming, jupyter n...",1.0,31.206190,"[star and snowflake schema, jupyter notebooks,..."
818,Python Project for Data Engineering,IBM,Course,1 - 4 Weeks,4.6,473.0,intermediate,https://www.coursera.org/learn/python-project-...,"32,522","['Information Engineering', 'Python Programmin...",['Demonstrate your skills in Python for workin...,Showcase your Python skills in this Data Engin...,"information engineering, python programming, e...",1.1,31.175709,"[python programming, extract transform and loa..."
711,Microsoft Power BI Data Analyst,Microsoft,Professional Certificate,3 - 6 Months,4.6,735.0,beginner,https://www.coursera.org/professional-certific...,"37,537","['Microsoft Excel', 'Data Analysis', 'power bi...",['Learn to use Power BI to connect to data sou...,Learners who complete this program will receiv...,"microsoft excel, data analysis, power bi, powe...",1.0,30.365659,"[microsoft excel, sql, power query, data analy..."
69,Applied Software Engineering Fundamentals,IBM,Specialization,3 - 6 Months,4.5,792.0,beginner,https://www.coursera.org/specializations/softw...,"3,076","['Linux', 'Software Design and Architecture', ...",['Perform basic R programming tasks like worki...,This Specialization is intended for anyone wit...,"linux, software design and architecture, pytho...",1.0,30.041204,"[python programming, software design and archi..."
830,Regression Analysis: Simplify Complex Data Rel...,Google,Course,1 - 3 Months,4.7,175.0,advanced,https://www.coursera.org/learn/regression-anal...,"17,632","['Predictive Modelling', 'Statistical Analysis...","['Investigate relationships in datasets', 'Ide...",This is the fifth of seven courses in the Goog...,"predictive modelling, statistical analysis, py...",1.2,29.161530,"[python programming, statistical analysis, pre..."
821,Python for Cybersecurity,Infosec,Specialization,3 - 6 Months,4.5,328.0,intermediate,https://www.coursera.org/specializations/pytho...,"12,936","['pre-

In [37]:
recommendations[
    [
        "course_title",
        "course_organization",
        "course_rating",
        "course_reviews_num",
        "course_difficulty",
        "course_rank"
    ]
]

,course_title,course_organization,course_rating,course_reviews_num,course_difficulty,course_rank
272,Django for Everybody,University of Michigan,4.7,904.0,intermediate,35.197024
991,用 Python 做商管程式設計（一）(Programming for Business C...,National Taiwan University,4.9,814.0,beginner,32.845622
933,The Power of Statistics,Google,4.8,270.0,advanced,32.268204
90,Automate Cybersecurity Tasks with Python,Google,4.7,915.0,beginner,32.054077
91,"BI Foundations with SQL, ETL and Data Warehousing",IBM,4.8,665.0,beginner,31.206190
818,Python Project for Data Engineering,IBM,4.6,473.0,intermediate,31.175709
711,Microsoft Power BI Data Analyst,Microsoft,4.6,735.0,beginner,30.365659
69,Applied Software Engineering Fundamentals,IBM,4.5,792.0,beginner,30.041204
830,Regression Analysis: Simplify Complex Data Rel...,Google,4.7,175.0,advanced,29.161530
821,Python for Cybersecurity,Infosec,4.5,328.0,intermediate,28.690486


In [38]:
def determine_course_level(
    related_skill_count,
    course_difficulty
):

    course_difficulty = str(course_difficulty).lower()

    if related_skill_count < 2:
        return "Beginner"

    if course_difficulty == "advanced":
        return "Advanced"

    return "Intermediate"

In [39]:
student_related_skills = 1

recommendations["recommended_level"] = recommendations[
    "course_difficulty"
].apply(
    lambda difficulty: determine_course_level(
        student_related_skills,
        difficulty
    )
)

In [40]:
recommendations[
    [
        "course_title",
        "course_organization",
        "course_difficulty",
        "recommended_level",
        "course_rating",
        "course_reviews_num",
        "course_rank"
    ]
]

,course_title,course_organization,course_difficulty,recommended_level,course_rating,course_reviews_num,course_rank
272,Django for Everybody,University of Michigan,intermediate,Beginner,4.7,904.0,35.197024
991,用 Python 做商管程式設計（一）(Programming for Business C...,National Taiwan University,beginner,Beginner,4.9,814.0,32.845622
933,The Power of Statistics,Google,advanced,Beginner,4.8,270.0,32.268204
90,Automate Cybersecurity Tasks with Python,Google,beginner,Beginner,4.7,915.0,32.054077
91,"BI Foundations with SQL, ETL and Data Warehousing",IBM,beginner,Beginner,4.8,665.0,31.206190
818,Python Project for Data Engineering,IBM,intermediate,Beginner,4.6,473.0,31.175709
711,Microsoft Power BI Data Analyst,Microsoft,beginner,Beginner,4.6,735.0,30.365659
69,Applied Software Engineering Fundamentals,IBM,beginner,Beginner,4.5,792.0,30.041204
830,Regression Analysis: Simplify Complex Data Rel...,Google,advanced,Beginner,4.7,175.0,29.161530
821,Python for Cybersecurity,Infosec,intermediate,Beginner,4.5,328.0,28.690486


In [41]:
courses_final = courses.sort_values(
    "course_rank",
    ascending=False
).reset_index(drop=True)

In [42]:
courses_final = courses_final[
    [
        "course_title",
        "course_organization",
        "course_certificate_type",
        "course_time",
        "course_rating",
        "course_reviews_num",
        "course_difficulty",
        "course_url",
        "course_students_enrolled",
        "course_skills_clean",
        "course_summary",
        "course_description",
        "course_rank"
    ]
]

In [43]:
courses_curated = courses_final.head(100).copy()

print("Final curated courses:", len(courses_curated))

Final curated courses: 100


In [44]:
courses_curated.head(10)

,course_title,course_organization,course_certificate_type,course_time,course_rating,course_reviews_num,course_difficulty,course_url,course_students_enrolled,course_skills_clean,course_summary,course_description,course_rank
0,Social and Economic Networks: Models and Anal...,Stanford University,Course,1 - 3 Months,4.8,712.0,advanced,https://www.coursera.org/learn/social-economic...,"67,579","social network, game theory, network analysis,...",[],Learn how to model social and economic network...,37.840213
1,Customer Experiences with Contact Center AI - ...,Google Cloud,Specialization,3 - 6 Months,4.9,936.0,intermediate,https://www.coursera.org/specializations/custo...,"30,099",,[],"In this course, you will:\n• Compare Functiona...",36.882063
2,Linux and Private Cloud Administration on IBM ...,Red Hat,Specialization,3 - 6 Months,4.8,921.0,intermediate,https://www.coursera.org/specializations/linux...,"10,280","system administration, red hat enterprise linu...",[],This specialization introduces Red Hat Enterpr...,36.044159
3,Essentials in Clinical Simulations Across the ...,The George Washington University,Course,1 - 3 Months,4.8,873.0,intermediate,https://www.coursera.org/learn/clinicalsimulat...,"46,526","nursing, healthcare, debriefing, inacsl standa...",[],This course covers in the chronological order ...,35.761864
4,U.S. Federal Taxation,University of Illinois at Urbana-Champaign,Specialization,3 - 6 Months,4.8,820.0,intermediate,https://www.coursera.org/specializations/unite...,"15,895","depreciation, federal tax returns, tax law",[],This Specialization introduces the U.S. federa...,35.431562
5,Healthcare Law,University of Pennsylvania,Specialization,3 - 6 Months,4.7,943.0,intermediate,https://www.coursera.org/specializations/healt...,"14,962","affordable care act, health law, health insura...",[],Are you looking to get your foot in the door a...,35.415152
6,Troubles du spectre de l'autisme : diagnostic,University of Geneva,Course,1 - 3 Months,4.9,694.0,intermediate,https://www.coursera.org/learn/troubles-spectr...,"22,526",,[],Qu’est-ce que l’autisme ?\nComment reconnaître...,35.271685
7,Django for Everybody,University of Michigan,Specialization,3 - 6 Months,4.7,904.0,intermediate,https://www.coursera.org/specializations/django,"50,277","cascading style sheets (css), html, javascript...",[],Diversity is a fact. It is also paradoxical. W...,35.197024
8,IBM Mainframe Developer,IBM,Professional Certificate,3 - 6 Months,4.6,932.0,intermediate,https://www.coursera.org/professional-certific...,"3,063","mainframe, z/os, enterprise software, security...",[],Gain the job-ready skills for an entry-level m...,34.602330
9,Business Data Management and Communication,University of Illinois at Urbana-Champaign,Specialization,3 - 6 Months,4.6,890.0,intermediate,https://www.coursera.org/specializations/busin...,"11,336","asset management, business analytics, financia...",[],"Our world has become increasingly digital, and...",34.369263


In [47]:
courses_curated.to_csv(
    "courses_processed.csv",
    index=False
)

print("courses_processed.csv created successfully!")

courses_processed.csv created successfully!


In [49]:
courses_json = courses_curated.to_dict(
    orient="records"
)

with open(
    "courses.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        courses_json,
        file,
        indent=4,
        ensure_ascii=False
    )

print("courses.json created successfully!")
print("Number of records:", len(courses_json))

courses.json created successfully!
Number of records: 100


In [50]:
with open(
    "courses.json",
    "r",
    encoding="utf-8"
) as file:

    final_courses = json.load(file)

print("Number of courses in JSON:", len(final_courses))

Number of courses in JSON: 100


In [51]:
final_courses[0]

{'course_title': 'Social and Economic Networks:  Models and Analysis',
 'course_organization': 'Stanford University',
 'course_certificate_type': 'Course',
 'course_time': '1 - 3 Months',
 'course_rating': 4.8,
 'course_reviews_num': 712.0,
 'course_difficulty': 'advanced',
 'course_url': 'https://www.coursera.org/learn/social-economic-networks',
 'course_students_enrolled': '67,579',
 'course_skills_clean': 'social network, game theory, network analysis, network theory',
 'course_summary': '[]',
 'course_description': 'Learn how to model social and economic networks and their impact on human behavior.  How do networks form, why do they exhibit certain patterns, and how does their structure impact diffusion, learning, and other behaviors?   We will bring together models and techniques from economics, sociology, math, physics, statistics and computer science to answer these questions.\nThe course begins with some empirical background on social and economic networks, and an overview of c

In [52]:
print("========== COURSE DATASET COMPLETE ==========")
print("Raw courses:", len(df))
print("Processed courses:", len(courses_curated))
print("Indexed terms:", len(inverted_index))
print("JSON records:", len(final_courses))
print("Output 1: courses_processed.csv")
print("Output 2: courses.json")

========== COURSE DATASET COMPLETE ==========
Raw courses: 1000
Processed courses: 100
Indexed terms: 2583
JSON records: 100
Output 1: courses_processed.csv
Output 2: courses.json


In [53]:
import os

print("courses_processed.csv exists:",
      os.path.exists("courses_processed.csv"))

print("courses.json exists:",
      os.path.exists("courses.json"))

courses_processed.csv exists: True
courses.json exists: True


In [54]:
print("JSON file size:",
      round(os.path.getsize("courses.json") / 1024, 2),
      "KB")

JSON file size: 213.61 KB


In [55]:
## pred or recommdetion

missing_skills = ["python"]

recommendations = recommend_courses(
    missing_skills,
    top_n=10
)

recommendations[
    [
        "course_title",
        "course_organization",
        "course_difficulty",
        "course_rating",
        "course_reviews_num",
        "course_rank"
    ]
]

,course_title,course_organization,course_difficulty,course_rating,course_reviews_num,course_rank
991,用 Python 做商管程式設計（一）(Programming for Business C...,National Taiwan University,beginner,4.9,814.0,32.845622
933,The Power of Statistics,Google,advanced,4.8,270.0,32.268204
90,Automate Cybersecurity Tasks with Python,Google,beginner,4.7,915.0,32.054077
91,"BI Foundations with SQL, ETL and Data Warehousing",IBM,beginner,4.8,665.0,31.206190
818,Python Project for Data Engineering,IBM,intermediate,4.6,473.0,31.175709
69,Applied Software Engineering Fundamentals,IBM,beginner,4.5,792.0,30.041204
830,Regression Analysis: Simplify Complex Data Rel...,Google,advanced,4.7,175.0,29.161530
821,Python for Cybersecurity,Infosec,intermediate,4.5,328.0,28.690486
473,IBM AI Enterprise Workflow,IBM,advanced,4.3,258.0,28.673233
536,Introducción a la programación en Python I: Ap...,Pontificia Universidad Católica de Chile,beginner,4.6,496.0,28.559514


In [56]:
missing_skills = ["python", "machine learning", "sql"]

recommendations = recommend_courses(
    missing_skills,
    top_n=10
)

recommendations[
    [
        "course_title",
        "course_organization",
        "course_difficulty",
        "course_rating",
        "course_reviews_num",
        "course_rank"
    ]
]

,course_title,course_organization,course_difficulty,course_rating,course_reviews_num,course_rank
272,Django for Everybody,University of Michigan,intermediate,4.7,904.0,35.197024
991,用 Python 做商管程式設計（一）(Programming for Business C...,National Taiwan University,beginner,4.9,814.0,32.845622
933,The Power of Statistics,Google,advanced,4.8,270.0,32.268204
90,Automate Cybersecurity Tasks with Python,Google,beginner,4.7,915.0,32.054077
91,"BI Foundations with SQL, ETL and Data Warehousing",IBM,beginner,4.8,665.0,31.206190
818,Python Project for Data Engineering,IBM,intermediate,4.6,473.0,31.175709
711,Microsoft Power BI Data Analyst,Microsoft,beginner,4.6,735.0,30.365659
69,Applied Software Engineering Fundamentals,IBM,beginner,4.5,792.0,30.041204
830,Regression Analysis: Simplify Complex Data Rel...,Google,advanced,4.7,175.0,29.161530
821,Python for Cybersecurity,Infosec,intermediate,4.5,328.0,28.690486
